# Try Qwen3-8B-Base

Use the repository `.venv` kernel and a GPU. Run the setup once, then edit the sentence. Prompts use the shared five few-shot examples and plain text, without a chat template.


In [ ]:
import csv
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent

with (ROOT / "data/active_pass_long/corpus.csv").open(newline="", encoding="utf-8") as handle:
    rows = [row for row in csv.DictReader(handle) if row["split"] in {"test", "gen_test"}]

from ac_tpr.dataset import TOKENIZER_ID, TOKENIZER_REVISION
from ac_tpr.prompts import PROMPT_TEMPLATE


In [ ]:
MODEL_ID = TOKENIZER_ID
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=TOKENIZER_REVISION)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=TOKENIZER_REVISION, dtype=torch.bfloat16, attn_implementation="eager",
).to(DEVICE).eval()
model.requires_grad_(False)


## Generate an answer

Set `EXAMPLE` to a row index, or replace `SENTENCE` with a pasted dataset sentence. The corresponding dataset answer is `rows[EXAMPLE]["agent"]`. Generation stops at EOS or 64 new tokens.

In [ ]:
EXAMPLE = 0
SENTENCE = rows[EXAMPLE]["sentence"]  # Or paste a dataset sentence here.

prompt = PROMPT_TEMPLATE.format(sentence=SENTENCE)
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(DEVICE)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
